In [4]:
import json
import pandas as pd

In [13]:
# chuyển cột date sang kiểu datetime
def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        return pd.DataFrame(data)

df_price = load_json("../data/processed/price_history_processed.json")
df_price["date"] = pd.to_datetime(df_price["date"], dayfirst=True)
df_price = df_price[(df_price['date'] >= '2024-01-01') & (df_price['date'] <= '2025-12-31')]
df_price = df_price.sort_values(['symbol', 'date'])





Phản ứng thị trường (01/06/2025 - 07/10/2025)

In [18]:


# Lọc dữ liệu trong khoảng thời gian
mask = (df_price['date'] >= '2025-06-01') & (df_price['date'] <= '2025-10-07')
df_price_phase1 = df_price.loc[mask]
growth = df_price_phase1.groupby('symbol')['price_change_pct'].sum()
print(growth)
growth.to_csv('q1_growth.csv')

symbol
ACB        24.16
BID        12.90
CTG        31.02
EIB        15.74
EVF        32.54
HDB        38.96
KLB        50.78
LPB        52.59
MBB        40.54
MSB        35.56
NAB        11.56
OCB        35.37
SHB        43.29
SSB         6.16
STB        38.19
TCB        29.04
TPB        37.76
VAB        20.58
VCB         9.50
VIB        24.41
VNINDEX    24.15
VPB        51.82
Name: price_change_pct, dtype: float64


Biến động sau tin FTSE Russell (07/10/2025)

In [19]:
daily_returns = df_price.copy()

# tạo cột Period (CASE WHEN)
daily_returns["Period"] = daily_returns["date"].apply(
    lambda x: "Before_News" if x <= "2025-10-07" else "After_News"
)

# lọc ngày (WHERE)
daily_returns = daily_returns[
    (daily_returns["date"] >= "2025-06-01") &
    (daily_returns["date"] <= "2025-12-31")
]

# GROUP BY + AVG
result = (
    daily_returns
    .groupby(["symbol", "Period"])
    .agg(
        Avg_Volume=("volume", "mean"),
        Avg_Daily_Return=("price_change_pct", "mean")
    )
    .reset_index()
)

print(result)
result.to_csv('q2_result.csv', index=False)

     symbol       Period    Avg_Volume  Avg_Daily_Return
0       ACB   After_News  1.030144e+07         -0.152951
1       ACB  Before_News  1.601014e+07          0.268444
2       BID   After_News  2.746957e+06         -0.036885
3       BID  Before_News  7.260387e+06          0.143333
4       CTG   After_News  8.998079e+06          0.033279
5       CTG  Before_News  9.648673e+06          0.344667
6       EIB   After_News  6.896197e+06         -0.273115
7       EIB  Before_News  1.610557e+07          0.174889
8       EVF   After_News  4.685161e+06         -0.338361
9       EVF  Before_News  1.779490e+07          0.361556
10      HDB   After_News  1.922796e+07          0.375410
11      HDB  Before_News  1.776266e+07          0.432889
12      KLB   After_News  4.791557e+05          0.187049
13      KLB  Before_News  7.239378e+05          0.564222
14      LPB   After_News  2.362297e+06         -0.355082
15      LPB  Before_News  3.432036e+06          0.584333
16      MBB   After_News  2.691

Hiệu ứng Momentum (đà tăng trưởng) của nhóm ngân hàng có thực
sự vượt trội trong các giai đoạn tin tức hay không?

In [20]:
# Tách riêng VNIndex và nhóm Bank

mask_momentum = (df_price['date'] >= '2025-06-01') & (df_price['date'] <= '2025-12-31')
df_price_momentum = df_price.loc[mask_momentum]
# Tính return trung bình của nhóm Bank theo ngày
bank_return = df_price_momentum[df_price_momentum['symbol'] != 'VNINDEX'].groupby('date')['price_change_pct'].mean()
vni_return = df_price_momentum[df_price_momentum['symbol'] == 'VNINDEX'].set_index('date')['price_change_pct']
excess_return = bank_return - vni_return

# Gộp lại để so sánh
momentum_df_price = pd.DataFrame({'Bank_Momentum': bank_return, 'VNIndex_Momentum': vni_return, 'Excess Return': excess_return}).dropna()
print(momentum_df_price)
momentum_df_price.to_csv('q3_momentum.csv')

            Bank_Momentum  VNIndex_Momentum  Excess Return
date                                                      
2025-06-02       0.947143              0.28       0.667143
2025-06-03       0.303333              0.82      -0.516667
2025-06-04      -0.268095             -0.11      -0.158095
2025-06-05      -0.536190             -0.27      -0.266190
2025-06-06      -0.603333             -0.91       0.306667
...                   ...               ...            ...
2025-12-25      -1.433333             -2.24       0.806667
2025-12-26      -0.419048             -0.75       0.330952
2025-12-29       0.119524              1.45      -1.330476
2025-12-30       0.762857              0.69       0.072857
2025-12-31       0.178095              1.00      -0.821905

[151 rows x 3 columns]


Mức độ rủi ro thị trường (độ biến động, mức giảm tối đa,...)

In [21]:
# 1. Volatility (Độ biến động năm)

volatility = df_price.groupby('symbol')['price_change_pct'].std() * (250**0.5)
# 1. Chuyển % về số thập phân

df_price['daily_ret'] = df_price['price_change_pct'] / 100

# 2. Tính Chỉ số tích lũy (Equity Curve) cho từng mã
# Giả sử bắt đầu với 1 đồng, ta dùng cumprod()
df_price['equity_curve'] = df_price.groupby('symbol')['daily_ret'].transform(lambda x: (1 + x).cumprod())

# 3. Tính Đỉnh (Running Peak)
df_price['peak'] = df_price.groupby('symbol')['equity_curve'].transform(lambda x: x.cummax())

# 4. Tính Drawdown theo từng thời điểm
df_price['drawdown'] = (df_price['equity_curve'] - df_price['peak']) / df_price['peak']

# 5. Lấy Max Drawdown (giá trị nhỏ nhất)
max_dd = df_price.groupby('symbol')['drawdown'].min() * 100
risk_metrics = pd.DataFrame({'Volatility': volatility, 'Max_Drawdown': max_dd})
print("--- Max Drawdown chuẩn (đã điều chỉnh cổ tức/chia tách) ---")
print(risk_metrics)
risk_metrics.to_csv('q4_risk.csv')

--- Max Drawdown chuẩn (đã điều chỉnh cổ tức/chia tách) ---
         Volatility  Max_Drawdown
symbol                           
ACB       22.120937    -20.193585
BID       24.503879    -27.303883
CTG       27.257444    -20.745394
EIB       33.813059    -35.211045
EVF       41.377426    -55.983918
HDB       30.965585    -30.951691
KLB       42.860487    -30.290528
LPB       31.606085    -23.007157
MBB       26.915755    -19.673287
MSB       28.078781    -24.266252
NAB       25.104356    -17.608749
OCB       28.775438    -27.049392
SHB       28.675278    -19.667762
SSB       25.113384    -28.303742
STB       30.807232    -24.573361
TCB       28.579682    -22.521610
TPB       30.474050    -33.444986
VAB       35.892473    -26.947806
VCB       19.974050    -21.978871
VIB       24.477841    -27.318920
VNINDEX   17.650005    -18.105573
VPB       30.755303    -31.331491


In [ ]:
delta = df_price['price_change_pct']

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df_price['RSI'] = 100 - (100/(1+rs))
print(df_price[('symbol', 'RSI')])